# Fine-tune Gemma 4 E2B as a model router

This Colab QLoRA-tunes **Gemma 4 E2B Instruct** with **Unsloth** so it routes a software task to one model:

- `luna` = simple / bounded change
- `astra` = complex / security-sensitive / architectural work

The labeled examples, and the train/test split, are the same demo set as `ModernBERT_train_classify.ipynb`. ModernBERT learns a classification head. This notebook teaches Gemma to answer with the route name.

A free T4 is enough for 4-bit QLoRA. In Colab use **Runtime → Change runtime type → T4 GPU**.

Gemma weights are gated. Accept the license on [google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it) and sign in if the download returns 401.


In [5]:
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


## 1. Install the Gemma 4 / Unsloth stack

Same install as the Gemma 4 E2B tweet notebook, so the chat template and 4-bit loader match current Unsloth Gemma 4 notebooks.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -U unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10':'0.0.34',
        '2.9':'0.0.33.post1',
        '2.8':'0.0.32.post2'
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm

import torch
torch._dynamo.config.recompile_limit = 64


## 2. Configuration


In [ ]:
# ---------- Model ----------
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

# ---------- Training ----------
LORA_R = 8
LORA_ALPHA = 8
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4
WARMUP_STEPS = 2
SEED = 3407

# Same train/test split as ModernBERT_train_classify.ipynb.
SPLIT_SEED = 42
TEST_SIZE = 0.25

ROUTER_SYSTEM_PROMPT = (
    "You are a model router for software tasks. "
    "Reply with only one route name: luna or astra.\n"
    "luna: a simple, bounded change such as copy, color, docs, tests, "
    "renames, logging, in-memory fields, or a helper that does not change behavior.\n"
    "astra: complex, security-sensitive, or architectural work such as "
    "authentication, tokens, encryption, authorization, database migrations, "
    "splitting services, or event-driven workflows."
)


## 3. Demo dataset

Same 32 tasks as the ModernBERT classifier notebook. `0` is `luna`, `1` is `astra`.

The split is stratified with `random_state=42`, matching that notebook. The paraphrases below are the differently worded checks from the classifier notebook, and they stay out of training.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

ID2ROUTE = {0: "luna", 1: "astra"}

examples = [
    # Luna: simple / bounded
    ("Fix a typo in the README.", 0),
    ("Change the primary button color from blue to green.", 0),
    ("Rename a local variable for clarity.", 0),
    ("Add a missing unit test for an existing helper.", 0),
    ("Update an API error message.", 0),
    ("Add pagination to an existing endpoint.", 0),
    ("Refactor duplicate validation logic into a shared helper.", 0),
    ("Add structured logging around failed HTTP requests.", 0),
    ("Update the OAuth login page copy without changing authentication logic.", 0),
    ("Rename the authentication middleware function without changing behavior.", 0),
    ("Add a runtime-only field to the User object that is not persisted.", 0),
    ("Add another field to an API response using data already in memory.", 0),
    ("Improve the wording of a validation error.", 0),
    ("Add a new test case for an existing endpoint.", 0),
    ("Document how to run the development server.", 0),
    ("Extract a small helper function without changing behavior.", 0),

    # Astra: complex / risky
    ("Implement Google OAuth login and persist users in PostgreSQL.", 1),
    ("Add refresh token rotation and revocation.", 1),
    ("Migrate user IDs from integers to UUIDs.", 1),
    ("Replace session authentication with JWT authentication.", 1),
    ("Add role-based access control for admins and normal users.", 1),
    ("Create an audit log table and record security-sensitive user actions.", 1),
    ("Add an encrypted API key field to the User model and persist it.", 1),
    ("Redesign the authorization layer across multiple services.", 1),
    ("Implement password reset using signed expiring tokens.", 1),
    ("Add multi-tenant permissions and migrate existing authorization data.", 1),
    ("Change token issuance rules and refresh-token storage.", 1),
    ("Move authentication state from database-backed sessions to stateless tokens.", 1),
    ("Add a new persisted field to User and write the required database migration.", 1),
    ("Introduce an event-driven workflow across the API and worker services.", 1),
    ("Split the monolith authentication module into separate services.", 1),
    ("Add end-to-end encryption for stored customer credentials.", 1),
]

df = pd.DataFrame(examples, columns=["text", "label"])
df["route"] = df["label"].map(ID2ROUTE)

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Held out of the labeled split. Expected routes follow the ModernBERT demo.
paraphrase_df = pd.DataFrame(
    [
        ("Please clean up the docs and correct a spelling mistake.", "luna"),
        ("Users should be able to sign in with Google and their linked identity must be stored.", "astra"),
        ("Add a computed property to User that only exists while the app is running.", "luna"),
        ("Change how refresh tokens are generated, stored, and revoked.", "astra"),
        ("Add one more assertion to the existing test suite.", "luna"),
    ],
    columns=["text", "route"],
)

print("Train:", len(train_df), "Test:", len(test_df))
print(train_df["route"].value_counts().to_dict())
train_df.head()


## 4. Load Gemma 4 E2B in 4-bit

Text-only LoRA. Vision and audio layers stay frozen.


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
)

print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


## 5. Render Gemma 4 conversations

The supervised target is only the route name. `train_on_responses_only` later masks the system and user turns, using Gemma 4 markers:

`<|turn>user` and `<|turn>model`.

Use the non-thinking `gemma-4` template. E2B should answer with the route, not a thought trace.


In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

def as_message(role, text):
    return {
        "role": role,
        "content": [{"type": "text", "text": text}],
    }

def make_conversation(text, route):
    return {
        "conversations": [
            as_message("system", ROUTER_SYSTEM_PROMPT),
            as_message("user", text),
            as_message("assistant", route),
        ]
    }

def to_dataset(frame):
    records = [
        make_conversation(row.text, row.route)
        for row in frame.itertuples(index=False)
    ]
    dataset = Dataset.from_list(records)

    def formatting_prompts_func(examples):
        texts = [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            ).removeprefix("<bos>")
            for convo in examples["conversations"]
        ]
        return {"text": texts}

    return dataset.map(formatting_prompts_func, batched=True)

train_dataset = to_dataset(train_df)

sample = train_dataset[0]["text"]
print(sample)
print()
assert "<|turn>user\n" in sample, "Missing Gemma 4 user turn marker."
assert "<|turn>model\n" in sample, "Missing Gemma 4 model turn marker."
model_span = sample.split("<|turn>model\n", 1)[-1].lstrip()
assert model_span.startswith(("luna", "astra")), model_span[:80]


## 6. Score the base model

Same prompt, before any LoRA update. Later cells reprint this so you can see whether fine-tuning beat the instruction model.


In [ ]:
import re

ROUTE_RE = re.compile(r"\b(luna|astra)\b", re.IGNORECASE)

def route_task(text, max_new_tokens=8):
    messages = [
        as_message("system", ROUTER_SYSTEM_PROMPT),
        as_message("user", text),
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    raw = tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    ).strip()
    match = ROUTE_RE.search(raw)
    return {
        "route": match.group(1).lower() if match else None,
        "raw": raw,
    }

def score_frame(frame, label):
    rows = []
    for row in frame.itertuples(index=False):
        pred = route_task(row.text)
        rows.append({
            "split": label,
            "text": row.text,
            "gold": row.route,
            "pred": pred["route"],
            "raw": pred["raw"],
            "correct": pred["route"] == row.route,
        })
    scored = pd.DataFrame(rows)
    accuracy = scored["correct"].mean()
    print(f"{label}: {scored['correct'].sum()}/{len(scored)} = {accuracy:.0%}")
    return scored

base_test = score_frame(test_df, "base holdout")
base_paraphrase = score_frame(paraphrase_df, "base paraphrase")
pd.concat([base_test, base_paraphrase], ignore_index=True)[
    ["split", "gold", "pred", "raw", "text"]
]


## 7. Add LoRA adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

model.print_trainable_parameters()


## 8. Train on the route name only

Loss is applied to the `<|turn>model` span, not the task text. The next cell prints that span. It should be the route, not the whole prompt.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir="gemma4-e2b-router-checkpoints",
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=1,
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

row = trainer.train_dataset[0]
trained = tokenizer.decode([x for x in row["labels"] if x != -100])
print("Loss is applied to:\n", repr(trained))
if not any(name in trained.lower() for name in ("luna", "astra")):
    raise RuntimeError(
        "Response masking dropped the route name. "
        "The Gemma 4 turn markers did not match the rendered sample."
    )


## 9. Train


In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Training examples: {len(train_dataset)}")
print(f"Epochs: {EPOCHS}")

trainer_stats = trainer.train()
trainer_stats


## 10. Plot loss

Training loss is logged every optimizer step. The route comparison in the next section is the actual check. Holdout loss is not a reliable score on 8 examples.


In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
plotted = False
for key, label in (("loss", "Training"),):
    points = [row for row in history if key in row and "step" in row]
    if not points:
        continue
    ax.plot(
        [row["step"] for row in points],
        [row[key] for row in points],
        marker="o",
        label=label,
    )
    plotted = True

ax.set(title="Gemma 4 E2B router", xlabel="Optimizer step", ylabel="Loss")
ax.grid(True, alpha=0.3)
if plotted:
    ax.legend()
else:
    ax.text(0.5, 0.5, "No loss logged", ha="center", va="center", transform=ax.transAxes)
plt.show()


## 11. Compare base vs fine-tuned routes


In [ ]:
tuned_test = score_frame(test_df, "tuned holdout")
tuned_paraphrase = score_frame(paraphrase_df, "tuned paraphrase")

def accuracy(frame):
    return float(frame["correct"].mean())

comparison = pd.DataFrame(
    [
        {"split": "holdout", "base": accuracy(base_test), "tuned": accuracy(tuned_test)},
        {"split": "paraphrase", "base": accuracy(base_paraphrase), "tuned": accuracy(tuned_paraphrase)},
    ]
)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.0%}"))
tuned = pd.concat([tuned_test, tuned_paraphrase], ignore_index=True)
tuned[["split", "gold", "pred", "correct", "raw", "text"]]


## 12. Route a new task


In [ ]:
route_task(
    "Implement OAuth authentication and add a PostgreSQL migration for refresh tokens."
)


## 13. Save the LoRA adapter

This is the small artifact to keep. Loading it later still requires Gemma 4 E2B.


In [ ]:
OUTPUT_DIR = "gemma4-e2b-model-router-lora"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -qr {OUTPUT_DIR}.zip {OUTPUT_DIR}

try:
    from google.colab import files
    files.download(f"{OUTPUT_DIR}.zip")
except Exception as exc:
    print(f"Saved {OUTPUT_DIR}/ (download skipped: {exc})")


## 14. Optional exports

Merged weights and GGUF use much more disk than the adapter. Uncomment only what you need.


In [ ]:
# MERGED_DIR = "gemma4-e2b-model-router-merged"
# model.save_pretrained_merged(MERGED_DIR, tokenizer)

# GGUF_DIR = "gemma4-e2b-model-router-gguf"
# model.save_pretrained_gguf(
#     GGUF_DIR,
#     tokenizer,
#     quantization_method="Q8_0",
# )

# from huggingface_hub import notebook_login
# notebook_login()
# repo_id = "YOUR_USERNAME/gemma4-e2b-model-router"
# model.push_to_hub(repo_id)
# tokenizer.push_to_hub(repo_id)


## 15. Release GPU memory

Run this last. It drops the model, trainer, and tokenizer, then returns the CUDA cache to the runtime. The LoRA directory and zip on disk are left alone. Route anything else only after loading the adapter again.


In [ ]:
import gc

def reserved_gb():
    if not torch.cuda.is_available():
        return None
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / 1024**3

before = reserved_gb()

for name in ("trainer", "model", "tokenizer"):
    obj = globals().pop(name, None)
    if obj is None:
        continue
    try:
        obj.to("cpu")
    except Exception:
        pass
    del obj

gc.collect()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

after = reserved_gb()
if before is None:
    print("No CUDA device. Dropped model, trainer, and tokenizer.")
else:
    print(f"Reserved GPU memory: {before:.2f} GB -> {after:.2f} GB")


## What this demo does not show

32 examples is enough to check that Gemma emits `luna` or `astra`. It is not enough to trust the route.

For a real router:

1. Collect **500–5,000+** labeled tasks.
2. Keep paraphrases of the same task in the same split.
3. Add hard negatives: wording about auth or migrations that is actually a bounded change, and the reverse.
4. Compare the fine-tune with the base model on unseen wording before you keep it.
5. Prefer a small encoder when you only need the label. Use this Gemma route when the router itself should be the language model.

### References

- Gemma 4 in Unsloth: https://unsloth.ai/docs/models/gemma-4/train
- Unsloth Gemma 4 E2B: https://huggingface.co/unsloth/gemma-4-E2B-it
- Matching classifier notebook: `colab/ModernBERT_train_classify.ipynb`
